# AI Robotics – Referenční řešení (notebook)

Tento notebook krok za krokem ukazuje, jak vytvořit soutěžní `submission.csv` pro úlohu **AI Robotics – Rozpoznání strategie autonomního robota**.

Pokryté části:
- diagnostika dat (`train.csv`, `test.csv`),
- přesné výpočty pro Úkoly 1–4,
- robustní klasifikační pipeline pro Úkol 5,
- porovnání modelů přes **Macro-F1**,
- trénink finálního modelu a export `submission.csv` + `metrics.json`.

## 1) Nastavení a importy

Notebook je deterministický (`RANDOM_STATE=42`) a používá standardní knihovny pro tabulková data.

In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

RANDOM_STATE = 42
ALLOWED_STRATEGIES = {"explorer", "collector", "guardian", "sprinter"}
TARGET_COL = "strategy_label"
ID_COL = "robot_id"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 2) Načtení dat

> Pokud notebook běží v Colabu, nahrajte `train.csv` a `test.csv` do aktuální pracovní složky,
> nebo změňte `DATA_DIR` níže.

In [ ]:
DATA_DIR = Path(".")
OUTPUT_DIR = Path(".")

train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(f"Expected train.csv and test.csv in {DATA_DIR.resolve()}")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("train shape:", train_df.shape)
print("test shape: ", test_df.shape)

## 3) Kontroly struktury dat

Ověříme:
- že `strategy_label` je pouze v tréninku,
- že existuje `robot_id` v obou souborech,
- datové typy, missing values a distribuci tříd.

In [ ]:
assert TARGET_COL in train_df.columns, f"Missing {TARGET_COL} in train.csv"
assert TARGET_COL not in test_df.columns, f"{TARGET_COL} must not be in test.csv"
assert ID_COL in train_df.columns and ID_COL in test_df.columns, "robot_id must exist in both files"

invalid_labels = set(train_df[TARGET_COL].dropna().unique()) - ALLOWED_STRATEGIES
assert not invalid_labels, f"Unexpected labels: {sorted(invalid_labels)}"

print("
Train columns:")
print(train_df.columns.tolist())

print("
Test columns:")
print(test_df.columns.tolist())

print("
Missing values (train):")
print(train_df.isna().sum().sort_values(ascending=False).head(20))

print("
Missing values (test):")
print(test_df.isna().sum().sort_values(ascending=False).head(20))

print("
Class distribution:")
print(train_df[TARGET_COL].value_counts())

## 4) Přesné řešení Úkolů 1–4

In [ ]:
required_cols = ["arena_type", "avg_speed_mps", "items_collected"]
missing_required = [c for c in required_cols if c not in train_df.columns]
if missing_required:
    raise ValueError(f"Missing columns for subtasks 1-4: {missing_required}")

subtasks_1_to_4 = {
    1: int(train_df["arena_type"].nunique(dropna=True)),
    2: float(train_df["avg_speed_mps"].max()),
    3: str(train_df["arena_type"].mode(dropna=True).iloc[0]),
    4: float(train_df["items_collected"].max()),
}

subtasks_1_to_4

## 5) Příprava pipeline pro Úkol 5

Použijeme tři kandidátní modely:
- Logistic Regression,
- Random Forest,
- HistGradientBoosting.

Vyhodnocení: `StratifiedKFold` + `macro F1`.

In [ ]:
X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL].astype(str)

numeric_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

cat_ohe_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

cat_ord_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocess_ohe = ColumnTransformer([
    ("num", num_pipe, numeric_cols),
    ("cat", cat_ohe_pipe, categorical_cols),
])

preprocess_ord = ColumnTransformer([
    ("num", num_pipe, numeric_cols),
    ("cat", cat_ord_pipe, categorical_cols),
])

pipelines = {
    "logreg": Pipeline([
        ("prep", preprocess_ohe),
        ("clf", LogisticRegression(max_iter=1500, random_state=RANDOM_STATE, multi_class="multinomial")),
    ]),
    "random_forest": Pipeline([
        ("prep", preprocess_ohe),
        ("clf", RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "hist_gb": Pipeline([
        ("prep", preprocess_ord),
        ("clf", HistGradientBoostingClassifier(learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=20, random_state=RANDOM_STATE)),
    ]),
}

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

## 6) Cross-validace přes Macro-F1

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, pipe in pipelines.items():
    fold_scores = []
    for tr_idx, va_idx in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model = clone(pipe)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_va)
        fold_scores.append(float(f1_score(y_va, pred, average="macro")))

    cv_results[name] = {
        "fold_macro_f1": fold_scores,
        "mean_macro_f1": float(np.mean(fold_scores)),
        "std_macro_f1": float(np.std(fold_scores)),
    }

pd.DataFrame({k: {"mean_macro_f1": v["mean_macro_f1"], "std_macro_f1": v["std_macro_f1"]} for k, v in cv_results.items()}).T.sort_values("mean_macro_f1", ascending=False)

## 7) Finální trénink, predikce a `submission.csv`

In [ ]:
best_model_name = max(cv_results, key=lambda k: cv_results[k]["mean_macro_f1"])
best_model = clone(pipelines[best_model_name])
best_model.fit(X_train, y_train)

test_pred = best_model.predict(test_df)

invalid_pred = set(pd.Series(test_pred).unique()) - ALLOWED_STRATEGIES
if invalid_pred:
    raise ValueError(f"Invalid predicted labels: {sorted(invalid_pred)}")

global_rows = pd.DataFrame([
    {"robot_id": "GLOBAL", "subtaskID": 1, "answer": str(subtasks_1_to_4[1])},
    {"robot_id": "GLOBAL", "subtaskID": 2, "answer": str(subtasks_1_to_4[2])},
    {"robot_id": "GLOBAL", "subtaskID": 3, "answer": str(subtasks_1_to_4[3])},
    {"robot_id": "GLOBAL", "subtaskID": 4, "answer": str(subtasks_1_to_4[4])},
])

task5_rows = pd.DataFrame({
    "robot_id": test_df[ID_COL].astype(str),
    "subtaskID": 5,
    "answer": test_pred.astype(str),
})

submission = pd.concat([global_rows, task5_rows], ignore_index=True)
assert submission.columns.tolist() == ["robot_id", "subtaskID", "answer"]
assert len(submission[(submission.robot_id == "GLOBAL") & (submission.subtaskID.isin([1,2,3,4]))]) == 4
assert len(submission[submission.subtaskID == 5]) == len(test_df)
assert set(submission[submission.subtaskID == 5]["answer"].unique()).issubset(ALLOWED_STRATEGIES)

submission_path = OUTPUT_DIR / "submission.csv"
metrics_path = OUTPUT_DIR / "metrics.json"

submission.to_csv(submission_path, index=False)

metrics = {
    "best_model": best_model_name,
    "cv_results": cv_results,
    "subtasks_1_to_4": subtasks_1_to_4,
}
metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")

print("Best model:", best_model_name)
print("Saved:", submission_path.resolve())
print("Saved:", metrics_path.resolve())
submission.head(10)

## 8) Shrnutí

- Úkoly 1–4 jsou spočítané přímo z `train.csv`.
- Úkol 5 používá férovou validaci přes `Macro-F1` a výběr nejlepšího modelu.
- Výstup `submission.csv` splňuje soutěžní formát:
  - 4 globální řádky (`GLOBAL`, subtask 1–4),
  - 1 řádek pro každý `robot_id` z `test.csv` se `subtaskID=5`.